# Week 7: In-Class Exercises
Due: noon Friday October 16th on HuskyCT  
Submit with filename: `w7_[LASTNAME].ipynb`

```{exercise} Ex. Orbital Conservation
:label: icc_7-1
:number: 1
We are going back to our 2D orbit problem from last week to study conservation properties of our various ODE methods. To do so, we'll want to run a 2D orbit simulation for a longer time: over multiple periods.
For practical purposes, we will start with the following functions: `rhs()` for computing $\dot{x},\dot{y},\dot{u},\dot{v}$ for a vector $[x,y,u,v]$, `Euler` an implementation of the Euler method, `RK4` an implementation of the 4th order Runge-Kutta method,  `EulerCromer` an implementation of the Euler-Cromer method. 
+ Write functions for calculating the relative change in energy $e$ and angular momentum $j$ for a given set of outputs from `Euler`,`Euler-Cromer`, or `RK4`
+ Run the simulations with the provided initial condition vector `vec0`: $[x_0, y_0, u_0, v_0]$ for at least 50 orbits this time with a $dt = 0.06$
+ Make a 1) plot of the orbit trajectory across both methods and 2) a plot of the relative change in energy and angular momentum (as a subplot) for the two methods to compare
```

In [ ]:
import numpy as np
from math import pi,sqrt
import matplotlib.pyplot as plt

GM = 4. * pi**2

def rhs(vec):
    """
    for a vector computes and returns the right hand side of the ODEs for xdot, ydot, udot, and vdot as a vector array
    """
    x,y,u,v = vec 
    # current magnitude of radius vector
    r = sqrt(x**2 + y**2)

    # position ODE
    xdot = u
    ydot = v

    # velocity ODE
    udot = -GM*x/r**3
    vdot = -GM*y/r**3
    
    return np.array([xdot, ydot, udot, vdot])

def Euler(vec0,tf,dt):
    """ Iteratively integrate using the Forward Euler method a vector of [x,y,u,v] to some final time tf with timesteps of size dt
    Returns:
    a list of histories for position and velocity at times t_vals """
    vec = vec0
    t_vals = [0]
    t = 0.0
    x_vals,y_vals = [vec0[0]],[vec0[1]]
    u_vals,v_vals = [vec0[2]],[vec0[3]]
    while t <= tf:
        vec_dot = rhs(vec)
        vec += dt * vec_dot
        t += dt
        x_vals.append(vec[0])
        y_vals.append(vec[1])
        u_vals.append(vec[2])
        v_vals.append(vec[3])
        t_vals.append(t)
    return x_vals, y_vals, u_vals, v_vals, t_vals


def EulerCromer(vec0,tf,dt):
    """ Iteratively integrate using the EulerCromer method a vector of [x,y,u,v] to some final time tf with timesteps of size dt
    Returns:
    a list of histories for position and velocity at times t_vals """
    vec = vec0
    t_vals = [0]
    t = 0.0
    x_vals,y_vals = [vec0[0]],[vec0[1]]
    u_vals,v_vals = [vec0[2]],[vec0[3]]
    while t <= tf:
        v_dot = rhs(vec)
        vec[-2:] += dt * v_dot[-2:] 
        x_dot = rhs(vec)
        vec[:-2] += dt * x_dot[:-2]
        t += dt
        x_vals.append(vec[0])
        y_vals.append(vec[1])
        u_vals.append(vec[2])
        v_vals.append(vec[3])
        t_vals.append(t)
    return x_vals, y_vals, u_vals, v_vals, t_vals

def RK4(vec0,tf,dt):
    """ Iteratively integrate using the RK4 method a vector of [x,y,u,v] to some final time tf with timesteps of size dt
    Returns:
    a list of histories for position and velocity at times t_vals """
    vec = vec0
    t_vals = [0]
    t = 0.0
    x_vals,y_vals = [vec0[0]],[vec0[1]]
    u_vals,v_vals = [vec0[2]],[vec0[3]]
    while t <= tf:
        vec_dot = rhs(vec)
        k1 = dt * vec_dot
        k2 = dt * rhs(vec+k1*0.5)
        k3 = dt * rhs(vec+k2*0.5)
        k4 = dt * rhs(vec+k3)
        vec += (k1+2*k2+2*k3+k4)/6
        t += dt
        x_vals.append(vec[0])
        y_vals.append(vec[1])
        u_vals.append(vec[2])
        v_vals.append(vec[3])
        t_vals.append(t)
    return x_vals, y_vals, u_vals, v_vals, t_vals


def compute_dE(x,y,u,v):
    """
    given a set of x, y, u, v values, computes the total specific energy relative to the initial value of the energy
    assumes x, y, u, v are outputs of integrator function where x, y, u, v are lists of values over time
    """
    return

def compute_dj(x,y,u,v):
    """
    given a set of x, y, u, v values, computes the total specific angular momentum relative to the initial value
    assumes x, y, u, v are outputs of integrator function where x, y, u, v are lists of values over time
    """
    return 

fig,ax = plt.subplots(1)
fig2,ax2 = plt.subplots(2,1,sharex=True,constrained_layout=True)
h = 0.06 
tf = 50 # run for 50 orbits
for method, name in zip([Euler,EulerCromer,RK4],["Euler","Euler-Cromer","RK4"]):
    vec0 = np.array([0,1,-sqrt(GM),0])
    x,y,u,v,t = method(vec0,tf,h)
    ax.plot(x,y,label=name)
    x = np.array(x)
    y = np.array(y)
    u = np.array(u)
    v = np.array(v)
    ax2[0].plot(t,compute_dE(x,y,u,v),label=name)
    ax2[1].plot(t,compute_dj(x,y,u,v),label=name,ls='dashed')


# mark the Sun
ax.scatter([0], [0], s=250, marker=(20, 1), color="k")
ax.scatter([0], [0], s=200, marker=(20, 1), color="y")

# draw the analytic solution
theta = np.linspace(0.0, 2.0*np.pi, 360)
ax.plot(np.cos(theta), np.sin(theta), ls=':', color='k',label='exact')


ax.set_xlabel('x [au]')
ax.set_ylabel('y [au]')

ax2[1].set_xlabel('t [yr]')
ax2[0].set_ylabel(r'$\Delta e/e_0$')
ax2[1].set_ylabel(r'$\Delta j/j_0$')

ax.set_title('Orbit Trajectory h=0.06')
ax2[0].set_title('Energy conservation h=0.06')
ax2[1].set_title('Momentum conservation h=0.06')

for a in ax2:
    a.set_yscale('symlog',linthresh=0.01)

ax.legend()
ax2[0].legend()

ax.axis('equal')
ax.set_xlim(-2,2)
ax.set_ylim(-2,2)




```{exercise} Verlet Method
:label: icc_7-2
:number: 2
Based on the provided implementations in @icc_7-1, write a function to compute the same integration using the Verlet method, add the Verlet method to the plots from above.
```

In [ ]:
def Verlet(vec0,tf,dt):
    """ Iteratively integrate using the Verlet method a vector of [x,y,u,v] to some final time tf with timesteps of size dt
    Returns:
    a list of histories for position and velocity at times t_vals """
    vec = vec0
    t_vals = [0]
    t = 0.0
    x_vals,y_vals = [vec0[0]],[vec0[1]]
    u_vals,v_vals = [vec0[2]],[vec0[3]]
    while t <= tf:
        ## your code here
        t += dt
        x_vals.append(vec[0])
        y_vals.append(vec[1])
        u_vals.append(vec[2])
        v_vals.append(vec[3])
        t_vals.append(t)
    return x_vals, y_vals, u_vals, v_vals, t_vals

fig,ax = plt.subplots(1)
fig2,ax2 = plt.subplots(2,1,sharex=True,constrained_layout=True)
h = 0.06
tf = 50
for method, name in zip([Euler,EulerCromer,RK4,Verlet],["Euler","Euler-Cromer","RK4","Verlet"]):
    vec0 = np.array([0,1,-sqrt(GM),0])
    x,y,u,v,t = method(vec0,tf,h)
    ax.plot(x,y,label=name)
    x = np.array(x)
    y = np.array(y)
    u = np.array(u)
    v = np.array(v)
    ax2[0].plot(t,compute_dE(x,y,u,v),label=name)
    ax2[1].plot(t,compute_dj(x,y,u,v),label=name,ls='dashed')


# mark the Sun
ax.scatter([0], [0], s=250, marker=(20, 1), color="k")
ax.scatter([0], [0], s=200, marker=(20, 1), color="y")

# draw the analytic solution
theta = np.linspace(0.0, 2.0*np.pi, 360)
ax.plot(np.cos(theta), np.sin(theta), ls=':', color='k',label='exact')


ax.set_xlabel('x [au]')
ax.set_ylabel('y [au]')

ax2[1].set_xlabel('t [yr]')
ax2[0].set_ylabel(r'$\Delta e/e_0$')
ax2[1].set_ylabel(r'$\Delta j/j_0$')

ax.set_title('Orbit Trajectory h=0.06')
ax2[0].set_title('Energy conservation h=0.06')
ax2[1].set_title('Momentum conservation h=0.06')

for a in ax2:
    a.set_yscale('symlog',linthresh=0.01)

ax.legend()
ax2[0].legend()

ax.axis('equal')
ax.set_xlim(-2,2)
ax.set_ylim(-2,2)


```{exercise} Shooting Method for Projectile Motion
:label: icc_7-3
:number: 3
Let's write a shooting method solver for the example trajectory problem using BCs:
$$ x(0) = 0 \ x(t_1) = L \\
y(0) = 0 \ y(t_1) = 0 $$

for $t_1 = 10$, $L=10$, using a timestep $dt=0.01$

Using the `rhs` and `RK4` functions below for a vector in $x_0, y_0, u_0, v_0$, write a function for the shooting method.
+ Pick a root-finding method and write an implementation for it here. (Even bisection should work fine in this case)
+ Make a plot of the trajectory in $y vs. x$ for your final results.

Note: while testing the shooting method iteration, you may want to set up a break condition for the maximum number of iterations to avoid waiting endlessly if there are bugs that cause your solution to never converge. 
(When working properly, the program shouldn't take more than a few seconds to run)

```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

def rhs(x,y,u,v):
    g = 9.81
    xdot = u
    ydot = v
    udot = 0.0
    vdot = -g
    return np.array([xdot, ydot, udot, vdot],dtype='float')

def RK4(vec0,tf,dt,history=False):
    vec = np.copy(vec0) #note the copy here to avoid overwriting, since we need the shooting method to decide which next iteration to adopt
    t_vals = [0]
    t = 0.0
    x_vals,y_vals = [vec0[0]],[vec0[1]]
    u_vals,v_vals = [vec0[2]],[vec0[3]]
    t_vals = np.arange(0,tf+dt,dt)
    for t in t_vals:
        vec_dot = rhs(*vec)
        k1 = dt * vec_dot
        k2 = dt * rhs(*(vec+k1*0.5))
        k3 = dt * rhs(*(vec+k2*0.5))
        k4 = dt * rhs(*(vec+k3))
        vec += (k1+2.0*k2+2.0*k3+k4)/6.0
        x_vals.append(vec[0])
        y_vals.append(vec[1])
        u_vals.append(vec[2])
        v_vals.append(vec[3])
    # note the additional options here on what to return 
    if history == True:
        return x_vals, y_vals, u_vals, v_vals, t_vals
    else:
        return x_vals[-1], y_vals[-1]


def Shooting(x0,y0,xf,yf,u0,v0,tf,dt,target):

    return #RK4(vec0,tf,dt,history=True) return the final answer given the best initial conditions found
    
   
x, y, u, v, t = Shooting(0,0,10,0,tf=10,dt=0.005,target=1e-10)

fig, ax = plt.subplots()
ax.plot(x,y)        
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Projectile motion: $u_0 =$ {:.2f} m/s, $v_0 = ${:.2f}m/s'.format(u[0],v[0]))

## Synthesis Question
*Discuss the following prompt in a new markdown cell below*

> Q: Based on your plot from @7-1, is velocity-verlet a time-symmetric method? Explain why.